# Parallel thinking: overfit one Alice/Bob trajectory

Portable restoration of the pretrain cube example, arranged like [PR #6](https://github.com/dvmazur/async_llm/pull/6).
Default models: **Qwen3-4B-Thinking-2507 teacher → Qwen3-1.7B student**, for a 16 GiB Ampere GPU.
This records a **new** teacher trajectory; it does not claim to recover the unavailable QwQ-32B trace.
The model names and GPU are configurable below.

Alice and Bob read each other's unfinished blocks. Every round issues two concurrent
one-token requests and **waits for both** before moving finished blocks into common history.
This preserves the original notebook's round barrier; agents do not run at independent speeds.
No periodic S1 reminder is inserted. A final continuation asks for a boxed answer.

The notebook uses this checkout's `minisgl.training` for visibility-aware replay,
block-relative RoPE, cross-entropy and evaluation. All model runs are separate processes,
so inference can initialize CUDA and GPU memory is released between stages.
Use the **AsyncLLM (uv)** kernel and Run All. Each execution creates a fresh run directory.

Qwen3-0.6B was tested first and answered 144 instead of 294; it is not the default student.

The saved run used CUDA ordinal 1, which maps to physical GPU 2 on imladris.
The setup cell now selects that same card by UUID to avoid ambiguous GPU ordering.


In [1]:
import os
import sys
import json
import subprocess
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import Markdown, display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "python/minisgl/training/replay.py").is_file())
DEMO = ROOT / "scripts/parallel_thinking"
sys.path.insert(0, str(DEMO))
os.environ.setdefault("CUDA_HOME", "/usr/local/cuda")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MAX_JOBS", "4")
os.environ.setdefault("HF_HUB_DISABLE_IMPLICIT_TOKEN", "1")
# Resolve the physical nvidia-smi index to UUID: CUDA's default ordinal order
# differs on this mixed Quadro/A4000 host.
GPU = os.environ.get("PARALLEL_GPU", "2")
GPU_UUID = subprocess.check_output(["nvidia-smi", f"--id={GPU}",
           "--query-gpu=uuid", "--format=csv,noheader"], text=True).strip()
assert GPU_UUID.startswith("GPU-") and "\n" not in GPU_UUID
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_UUID
TEACHER = os.environ.get("PARALLEL_TEACHER", str(ROOT / "models/Qwen3-4B-Thinking-2507"))
STUDENT = os.environ.get("PARALLEL_STUDENT", str(ROOT / "models/Qwen3-1.7B"))
MAX_STEPS = 1024
TRAIN_STEPS = 80
RUN = ROOT / "training_runs/parallel_thinking" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN.mkdir(parents=True, exist_ok=False)
print("Physical GPU:", GPU, "UUID:", GPU_UUID, "\nTeacher:", TEACHER, "\nStudent:", STUDENT, "\nRun:", RUN)
print(subprocess.check_output(["nvidia-smi", f"--id={GPU}",
      "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader"], text=True))


GPU: 1 
Teacher: /home/uskovev/async_llm/models/Qwen3-4B-Thinking-2507 
Student: /home/uskovev/async_llm/models/Qwen3-1.7B 
Run: /home/uskovev/async_llm/training_runs/parallel_thinking/20261101T115331740144Z


NVIDIA RTX A4000, 16376 MiB, 15472 MiB



In [2]:
def run_stage(name, arguments):
    free = int(subprocess.check_output(["nvidia-smi", f"--id={GPU}",
               "--query-gpu=memory.free", "--format=csv,noheader,nounits"], text=True).strip())
    if free < 14000:
        raise RuntimeError(f"GPU {GPU}: only {free} MiB free; select a free 16 GiB GPU")
    log = RUN / f"{name}.log"
    command = [sys.executable, "-u", str(DEMO / "parallel.py"), *map(str, arguments)]
    print("Starting", name, "— log:", log, flush=True)
    with log.open("w") as output:
        process = subprocess.Popen(command, cwd=ROOT, env=os.environ.copy(),
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in process.stdout:
            output.write(line)
            output.flush()
            if line.startswith(("round ", "initial:", "update ", "evaluation ", '{"answer"')):
                print(line.rstrip(), flush=True)
        returncode = process.wait()
    if returncode:
        raise RuntimeError(f"{name} failed ({returncode}). {log}\n" + log.read_text()[-6000:])

from parallel import expected_answer, compare_rollouts
EXPECTED = expected_answer()  # validation only; never inserted into a model prompt


/home/uskovev/async_llm/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Record and validate the teacher

The recorder labels the exact decode row which predicted each token. It records the
actual batches, ordered cache views, visible lengths, copies and clears. Prefill and
flush rows have target `-100`; no global next-token shift is used.


In [3]:
TRACE = RUN / "teacher_trace.json"
TEACHER_RESULT = RUN / "teacher_result.json"
run_stage("teacher", ["rollout", "--model", TEACHER, "--output", TEACHER_RESULT,
                      "--trace", TRACE, "--max-steps", MAX_STEPS])
teacher = json.loads(TEACHER_RESULT.read_text())
assert teacher["answer"] == EXPECTED, "Teacher answer is wrong: inspect before training"
assert teacher["completed_steps"] > 0
assert all(v > 0 for v in teacher["generated_tokens"].values())
print("Teacher answer:", teacher["answer"], "Completed steps:", teacher["completed_steps"])
display(Markdown(teacher["transcript"] + "\n\n" + teacher["answer_text"]))


Starting teacher — log: /home/uskovev/async_llm/training_runs/parallel_thinking/20261101T115331740144Z/teacher.log


round 128: 2 completed steps


round 256: 2 completed steps


round 384: 2 completed steps


round 512: 2 completed steps


round 640: 4 completed steps


round 768: 4 completed steps


round 896: 5 completed steps


round 1024: 6 completed steps


Teacher answer: 294.0 Completed steps: 6


**Problem:** Three vertices of a cube are $P=(7,12,10)$ , $Q=(8,8,1)$ , and $R=(11,3,9)$ . What is the surface area of the cube?

Round 1024/1024

### Past steps

**Alice [1]:** Hi, I'm Alice. Here's how we can collaborate: I'll compute the distances between the points to find the edge length of the cube. Let's start with PQ, QR, and RP.

**Bob [1]:** Hi, I'm Bob. I need to find the surface area of the cube given three vertices. First, I should find the distances between P, Q, and R. Let's compute PQ, QR, and RP.

**Bob [2]:** 1. Compute PQ: distance between P(7,12,10) and Q(8,8,1): sqrt((8-7)^2 + (8-12)^2 + (1-10)^2) = sqrt(1 + 16 + 81) = sqrt(98) = 7*sqrt(2)

**Alice [3]:** 2. Compute QR: distance between Q(8,8,1) and R(11,3,9): sqrt((11-8)^2 + (3-8)^2 + (9-1)^2) = sqrt(9 + 25 + 64) = sqrt(98) = 7*sqrt(2)
**Bob [3]:** 2. Compute QR: distance between Q(8,8,1) and R(11,3,9): sqrt((11-8)^2 + (3-8)^2 + (9-1)^2) = sqrt(9 + 25 + 64) = sqrt(98) = 7*sqrt(2)

**Alice [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Bob [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Alice [5]:** Wow, all three distances are equal! So PQ = QR = RP = 7*sqrt(2). That means these three points form an equilateral triangle. In a cube, three vertices that form an equilateral triangle must be such that each pair is connected by a face diagonal or a space diagonal? Wait, but in a cube, the face diagonal is s*sqrt(2) and space diagonal is s*sqrt(3), where s is the edge length.

**Alice [2]:** 1. Compute PQ: distance between P(7,12,10) and Q(8,8,1): sqrt((8-7)^2 + (8-12)^2 + (1-10)^2) = sqrt(1 + 16 + 81) = sqrt(98) = 7*sqrt(2)
**Bob [2]:** 1. Compute PQ: distance between P(7,12,10) and Q(8,8,1): sqrt((8-7)^2 + (8-12)^2 + (1-10)^2) = sqrt(1 + 16 + 81) = sqrt(98) = 7*sqrt(2)

**Alice [3]:** 2. Compute QR: distance between Q(8,8,1) and R(11,3,9): sqrt((11-8)^2 + (3-8)^2 + (9-1)^2) = sqrt(9 + 25 + 64) = sqrt(98) = 7*sqrt(2)
**Bob [3]:** 2. Compute QR: distance between Q(8,8,1) and R(11,3,9): sqrt((11-8)^2 + (3-8)^2 + (9-1)^2) = sqrt(9 + 25 + 64) = sqrt(98) = 7*sqrt(2)

**Alice [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Bob [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Alice [5]:** Wow, all three distances are equal! So PQ = QR = RP = 7*sqrt(2). That means these three points form an equilateral triangle. In a cube, three vertices that form an equilateral triangle must be such that each pair is connected by a face diagonal or a space diagonal? Wait, but in a cube, the face diagonal is s*sqrt(2) and space diagonal is s*sqrt(3), where s is the edge length.

**Alice [3]:** 2. Compute QR: distance between Q(8,8,1) and R(11,3,9): sqrt((11-8)^2 + (3-8)^2 + (9-1)^2) = sqrt(9 + 25 + 64) = sqrt(98) = 7*sqrt(2)

**Bob [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Alice [5]:** Wow, all three distances are equal! So PQ = QR = RP = 7*sqrt(2). That means these three points form an equilateral triangle. In a cube, three vertices that form an equilateral triangle must be such that each pair is connected by a face diagonal or a space diagonal? Wait, but in a cube, the face diagonal is s*sqrt(2) and space diagonal is s*sqrt(3), where s is the edge length.

**Bob [3]:** 2. Compute QR: distance between Q(8,8,1) and R(11,3,9): sqrt((11-8)^2 + (3-8)^2 + (9-1)^2) = sqrt(9 + 25 + 64) = sqrt(98) = 7*sqrt(2)

**Alice [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Bob [5]:** 4. Now, since all three distances are equal, they form an equilateral triangle. In a cube, the side length of the equilateral triangle formed by three vertices can be related to the edge length s of the cube. If the triangle is formed by three vertices where each pair is connected by a face diagonal, then the side length of the triangle would be s*sqrt(2). But if it's formed by a space diagonal, it would be s*sqrt(3). Wait, but here the side length is 7*sqrt(2), so if it's a face diagonal, then s*sqrt(2) = 7*sqrt(2) => s = 7. If it's a space diagonal, s*sqrt(3) = 7*sqrt(2) => s = 7*sqrt(2/3), which is not an integer, but the problem says it's a cube, so probably s is integer. So likely it's a face diagonal.



### Work in progress (others)

**Alice [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Bob [5]:** 4. Now, since all three distances are equal, they form an equilateral triangle. In a cube, the side length of the equilateral triangle formed by three vertices can be related to the edge length s of the cube. If<...>

**Bob [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Alice [5]:** Wow, all three distances are equal! So PQ = QR =<...>





Wait, given the limited time, I have to give an answer right now. Considering all my previous attempts, I have to conclude that the final answer is \boxed{294}

## Two untrained baselines

First, a **native-prompt capability check** requires that the untrained 1.7B can solve
the problem with its own chat template. Then a **matched-prompt baseline** uses exactly
the teacher's prompt IDs and is reported even if incorrect. The latter is the fair
before/after comparison for training; the former is not substituted for it.

This distinction matters here: the Thinking teacher template adds `<think>`, while
the 1.7B template used by the original formatter does not. No target response tokens
or publication events are supplied to either rollout. The model can repeat itself
even when the numeric answer is correct.


In [4]:
NATIVE_BASELINE_RESULT = RUN / "native_baseline_result.json"
run_stage("native_baseline", ["rollout", "--model", STUDENT,
                             "--output", NATIVE_BASELINE_RESULT, "--max-steps", MAX_STEPS])
native_baseline = json.loads(NATIVE_BASELINE_RESULT.read_text())
print("Native-prompt untrained answer:", native_baseline["answer"])
assert native_baseline["answer"] == EXPECTED, "The student did not pass its native-prompt capability check"

BASELINE_RESULT = RUN / "baseline_result.json"
run_stage("baseline", ["rollout", "--model", STUDENT, "--output", BASELINE_RESULT,
                       "--prompt-from", TEACHER_RESULT, "--max-steps", MAX_STEPS])
baseline = json.loads(BASELINE_RESULT.read_text())
print("Matched-prompt untrained answer:", baseline["answer"], "Completed steps:", baseline["completed_steps"])
display(Markdown(baseline["transcript"] + "\n\n" + baseline["answer_text"]))
print("Matched-prompt baseline correct:", baseline["answer"] == EXPECTED)


Starting native_baseline — log: /home/uskovev/async_llm/training_runs/parallel_thinking/20261101T115331740144Z/native_baseline.log


round 128: 4 completed steps


round 256: 6 completed steps


round 384: 11 completed steps


round 512: 12 completed steps


round 640: 14 completed steps


... (11 outputs left out)

round 1024: 48 completed steps


Matched-prompt untrained answer: 588.0 Completed steps: 48


**Problem:** Three vertices of a cube are $P=(7,12,10)$ , $Q=(8,8,1)$ , and $R=(11,3,9)$ . What is the surface area of the cube?

Round 1024/1024

### Past steps

**Alice [1]:** Hi, I'm Alice. Here's how we can collaborate on solving the problem. Let me start by understanding the problem.

**Bob [1]:** Hi, I'm Bob. I need to find the surface area of a cube given three vertices. Let me think about the steps needed.

**Bob [2]:**  I need to find the edge length of the cube first. To do that, I should find the distance between two of the given points and use that to determine the edge length. Let me calculate the distance between P and Q.

**Alice [2]:**  I see. The problem says that three vertices of a cube are given: P=(7,12,10), Q=(8,8,1), and R=(11,3,9). I need to find the surface area of the cube. First, I should find the edge length of the cube.

**Alice [3]:**  The distance between P and Q is sqrt[(8-7)^2 + (8-12)^2 + (1-10)^2] = sqrt[1 + 16 + 81] = sqrt[98]. So, the edge length is sqrt(98). But wait, is this the edge length of the cube?

**Bob [3]:**  I'll calculate the distance between P and Q. The distance formula in 3D is sqrt[(x2 - x1)^2 + (y2 - y1)^2 + (z2 - z1)^2]. So, that's sqrt[(8-7)^2 + (8-12)^2 + (1-10)^2] = sqrt[1 + 16 + 81] = sqrt[98]. So, the edge length is sqrt(98). But I need to confirm if this is the edge length of the cube.

**Alice [4]:**  Yes, that's correct. The distance between P and Q is sqrt(98). But wait, is this the edge length of the cube? I need to check if the three points form a cube. Let me check the distance between Q and R.

**Bob [4]:**  I think the distance between P and Q is the edge length of the cube. Let me calculate the distance between Q and R. The distance between Q=(8,8,1) and R=(11,3,9) is sqrt[(11-8)^2 + (3-8)^2 + (9-1)^2] = sqrt[9 + 25 + 64] = sqrt[98]. So, the edge length is sqrt(98). That seems consistent.

**Alice [5]:**  I'll calculate the distance between Q and R. The distance between Q=(8,8,1) and R=(11,3,9) is sqrt[(11-8)^2 + (3-8)^2 + (9-1)^2] = sqrt[9 + 25 + 64] = sqrt[98]. So, the edge length is sqrt(98). That's the same as the distance between P and Q. So, the edge length is sqrt(98).

**Bob [5]:**  So, the edge length is sqrt(98). Now, the surface area of a cube is 6 times the square of the edge length. So, the surface area would be 6 * (sqrt(98))^2 = 6 * 98 = 588. But wait, is this correct?

**Alice [6]:**  The surface area of a cube is 6 * (edge length)^2. So, the surface area would be 6 * (sqrt(98))^2 = 6 * 98 = 588. But I need to make sure that the edge length is indeed sqrt(98). Let me check if the distance between P and R is also sqrt(98).

**Bob [6]:**  I think that's correct. The surface area is 588. But I need to make sure that the three points are not on the same face or something else. Wait, but the problem says three vertices of a cube, so they should form a face of the cube. So, the edge length is sqrt(98), and the surface area is 6 * (sqrt(98))^2 = 588. So, I think that's the answer.

**Alice [7]:**  I think the three points are not on the same face. The distance between P and Q is sqrt(98), and between Q and R is also sqrt(98), so the edge length is sqrt(98). So, the surface area is 6 * (sqrt(98))^2 = 588. I think that's the answer.

**Bob [7]:**  I think that's correct. The surface area of the cube is 588. I'll check if there's another way to approach this problem to confirm.

**Alice [8]:**  I think I'm done. The surface area of the cube is 588. I'll check if there's any other way to confirm this.

**Bob [8]:**  I think I'm done. The surface area of the cube is 588. I'll check if there's any other way to confirm this.

**Alice [9]:**  I think I'm done. The surface area of the cube is 588. I'll check if there's any other way to confirm this.

**Bob [9]:**  I think I'm done. The surface area of the cube is 588. I'll check if there's any other way to confirm this.

**Alice [10]:**  I think I'm done. The surface area of the cube is 588. I'll check if there's any other way to confirm this.

**Bob [10]:**  I think I'm done. The surface area of the cube is 588. I'll check if there's any other way to confirm this.

**Alice [11]:**  I think I'm done. The surface area of the cube is 588. I'll check if there's any other way to confirm this.

**Bob [11]:**  I think I'm done. The surface area of the cube is 588. I'll check if there's any other way to confirm this.

**Alice [12]:**  I think I'm done. The surface area of the cube is 588. I'll check if there's any other way to confirm this.

**Bob [12]:**  I think I'm done. The surface area of the cube is 588. I'll check if there's any other way to confirm this.

**Alice [13]:**  I think I'm done. The surface area of the cube is 588. I'll check if there's any other way to confirm this.

**Bob [13]:**  I think I'm done. The surface area of the cube is 588. I'll check if there's any other way to confirm this.

**Alice [14]:**  I think I'm done. The surface area of the cube is 588. I'll check if there's any other way to confirm this.

**Bob [14]:**  I think I'm done. The surface area of the cube is 588. I'll check if there's any other way to confirm this.

**Alice [15]:**  I think I'm done. The surface area of the cube is 588. I'll check if there's any other way to confirm this.

**Bob [15]:**  I think I'm done. The surface area of the cube is 588. I'll check if there's any other way to confirm this.

**Alice [16]:**  I think I'm done. The surface area of the cube is 588. I'll check if there's any other way to confirm this.

**Bob [16]:**  I think I'm done. The surface area of the cube is 588. I'll check if there's any other way to confirm this.

**Alice [17]:**  I think I'm done. The surface area of the cube is 588. I'll check if there's any other way to confirm this.

**Bob [17]:**  I think I'm done. The surface area of the cube is 588. I'll check if there's any other way to confirm this.

**Alice [18]:**  I think I'm done. The surface area of the cube is 588. I'll check if there's any other way to confirm this.

**Bob [18]:**  I think I'm done. The surface area of the cube is 588. I'll check if there's any other way to confirm this.

**Alice [19]:**  I think I'm done. The surface area of the cube is 588. I'll check if there's any other way to confirm this.

**Bob [19]:**  I think I'm done. The surface area of the cube is 588. I'll check if there's any other way to confirm this.

**Alice [20]:**  I think I'm done. The surface area of the cube is 588. I'll check if there's any other way to confirm this.

**Bob [20]:**  I think I'm done. The surface area of the cube is 588. I'll check if there's any other way to confirm this.

**Alice [21]:**  I think I'm done. The surface area of the cube is 588. I'll check if there's any other way to confirm this.

**Bob [21]:**  I think I'm done. The surface area of the cube is 588. I'll check if there's any other way to confirm this.

**Alice [22]:**  I think I'm done. The surface area of the cube is 588. I'll check if there's any other way to confirm this.

**Bob [22]:**  I think I'm done. The surface area of the cube is 588. I'll check if there's any other way to confirm this.

**Alice [23]:**  I think I'm done. The surface area of the cube is 588. I'll check if there's any other way to confirm this.

**Bob [23]:**  I think I'm done. The surface area of the cube is 588. I'll check if there's any other way to confirm this.

**Alice [24]:**  I think I'm done. The surface area of the cube is 588. I'll check if there's any other way to confirm this.

**Bob [24]:**  I think I'm done. The surface area of the cube is 588. I'll check if there's any other way to confirm this.



### Work in progress (others)

**Alice [25]:**  I think I'm done. The surface area of the cube is 588. I'll check if there's any other way to confirm this<...>

**Bob [25]:**  I think I'm done. The<...>





Wait, given the limited time, I have to give an answer right now. Considering all my previous attempts, I have to conclude that the final answer is \boxed{588}

Matched-prompt baseline correct: False


## Train on this one trajectory

Forward and backward use `minisgl.training.replay` and `minisgl.training.sft`.
Replay reconstructs the recorded visibility and block-relative RoPE. Native HF RoPE
is neutralized in replay; the replay attention applies the required rotations.

To fit 1.7B in 16 GiB, FP32 master weights and AdamW states live on **CPU**.
Forward/backward use BF16 compute weights on GPU with gradient checkpointing.
Gradients are copied to CPU in FP32, clipped, and used to update the FP32 masters;
updated weights are copied back to GPU. The saved checkpoint contains FP32 masters.
This trades speed for GPU memory. Query/loss chunks of 64 reduce workspace without
truncating context or changing the target set.
Train all weights with lr `3e-5`, seed 7, gradient clipping 1.0. Stop at 100% target
accuracy and mean CE below 0.01, or the configured step limit.


In [5]:
TRAINING = RUN / "training"
run_stage("train", ["train", "--model", STUDENT, "--trace", TRACE,
                    "--output", TRAINING, "--steps", TRAIN_STEPS])
metrics = json.loads((TRAINING / "metrics.json").read_text())
print(json.dumps(metrics, indent=2))
assert metrics["memorized"], "Step limit reached without full memorization"
CHECKPOINT = TRAINING / "checkpoint"


Starting train — log: /home/uskovev/async_llm/training_runs/parallel_thinking/20261101T115331740144Z/train.log


initial: {'loss': 0.16998979449272156, 'accuracy': 0.9502923976608187}


update 1: loss=0.169990; grad_norm=11.5199


update 2: loss=0.089196; grad_norm=2.5489


update 3: loss=0.039943; grad_norm=2.8736


update 4: loss=0.017198; grad_norm=2.0647


... (11 outputs left out)

update 14: loss=0.001501; grad_norm=0.6448


update 15: loss=0.001749; grad_norm=0.6466


evaluation 15: {'loss': 0.00021080879378132522, 'accuracy': 1.0}


{
  "initial": {
    "loss": 0.16998979449272156,
    "accuracy": 0.9502923976608187
  },
  "final": {
    "loss": 0.00021080879378132522,
    "accuracy": 1.0
  },
  "history": [
    {
      "step": 0,
      "loss": 0.16998979449272156,
      "accuracy": 0.9502923976608187
    },
    {
      "step": 5,
      "loss": 0.0054430789314210415,
      "accuracy": 0.9990253411306043
    },
    {
      "step": 10,
      "loss": 0.0013085746904835105,
      "accuracy": 0.9990253411306043
    },
    {
      "step": 15,
      "loss": 0.00021080879378132522,
      "accuracy": 1.0
    }
  ],
  "steps": 15,
  "memorized": true,
  "first_gradient_norms": {
    "embedding": 2.09375,
    "first_attention": 0.3984375,
    "last_mlp": 0.52734375,
    "lm_head": 2.09375
  },
  "supervised_tokens": 2052,
  "trace_rows": 3229,
  "training_seconds": 571.6942050219513,
  "peak_cuda_gib": 7.062503814697266,
  "lr": 3e-05,
  "seed": 7,
  "model": "/home/uskovev/async_llm/models/Qwen3-1.7B",
  "checkpoint": "/hom

## Reload and evaluate

`evaluate` is defined in **`python/minisgl/training/sft.py`**. It evaluates the fixed,
teacher-forced trace. Loading the checkpoint in a new process checks the saved artifact;
it is separate from the independent generation check below.


In [6]:
RELOADED = RUN / "reloaded_evaluation.json"
run_stage("reload", ["evaluate", "--model", CHECKPOINT, "--trace", TRACE, "--output", RELOADED])
reloaded = json.loads(RELOADED.read_text())
print(reloaded)
assert reloaded["accuracy"] == 1.0 and reloaded["loss"] < 0.01
assert abs(reloaded["loss"] - metrics["final"]["loss"]) < 1e-6


Starting reload — log: /home/uskovev/async_llm/training_runs/parallel_thinking/20261101T115331740144Z/reload.log


{'loss': 0.00021080879378132522, 'accuracy': 1.0}


## Independent student generation

Compare answer correctness, decoded transcript, token IDs and the ordered publication
events `(round, worker, step, tokens)`. Publication means moving a finished block into
common history; unfinished blocks are already visible to the other agent.
Exact equality applies to this round-based algorithm, **not** wall-clock timing,
internal scheduler batches, different delays, or generalization to new problems.


In [7]:
STUDENT_RESULT = RUN / "student_result.json"
run_stage("student", ["rollout", "--model", CHECKPOINT, "--output", STUDENT_RESULT,
                      "--prompt-from", TEACHER_RESULT, "--max-steps", MAX_STEPS])
student = json.loads(STUDENT_RESULT.read_text())
comparison = compare_rollouts(teacher, student)
comparison["baseline_answer"] = baseline["answer"]
comparison["native_baseline_answer"] = native_baseline["answer"]
comparison["baseline_completed_steps"] = baseline["completed_steps"]
(RUN / "rollout_check.json").write_text(json.dumps(comparison, indent=2))
print(json.dumps(comparison, indent=2))
display(Markdown(student["transcript"] + "\n\n" + student["answer_text"]))
assert comparison["correct_answer"], "Free generation gave an incorrect answer"
print("Exact replay in free generation:", all(comparison[k] for k in
      ("same_transcript", "same_merge_events", "same_answer_text", "same_token_ids")))
print("All artifacts:", RUN)


Starting student — log: /home/uskovev/async_llm/training_runs/parallel_thinking/20261101T115331740144Z/student.log


round 128: 2 completed steps


round 256: 2 completed steps


round 384: 2 completed steps


round 512: 2 completed steps


round 640: 4 completed steps


... (2 outputs left out)

round 1024: 6 completed steps


{
  "answer": 294.0,
  "correct_answer": true,
  "same_transcript": true,
  "same_merge_events": true,
  "same_answer_text": true,
  "same_token_ids": true,
  "teacher_completed_steps": 6,
  "student_completed_steps": 6,
  "baseline_answer": 588.0,
  "native_baseline_answer": 294.0,
  "baseline_completed_steps": 48
}


**Problem:** Three vertices of a cube are $P=(7,12,10)$ , $Q=(8,8,1)$ , and $R=(11,3,9)$ . What is the surface area of the cube?

Round 1024/1024

### Past steps

**Alice [1]:** Hi, I'm Alice. Here's how we can collaborate: I'll compute the distances between the points to find the edge length of the cube. Let's start with PQ, QR, and RP.

**Bob [1]:** Hi, I'm Bob. I need to find the surface area of the cube given three vertices. First, I should find the distances between P, Q, and R. Let's compute PQ, QR, and RP.

**Bob [2]:** 1. Compute PQ: distance between P(7,12,10) and Q(8,8,1): sqrt((8-7)^2 + (8-12)^2 + (1-10)^2) = sqrt(1 + 16 + 81) = sqrt(98) = 7*sqrt(2)

**Alice [3]:** 2. Compute QR: distance between Q(8,8,1) and R(11,3,9): sqrt((11-8)^2 + (3-8)^2 + (9-1)^2) = sqrt(9 + 25 + 64) = sqrt(98) = 7*sqrt(2)
**Bob [3]:** 2. Compute QR: distance between Q(8,8,1) and R(11,3,9): sqrt((11-8)^2 + (3-8)^2 + (9-1)^2) = sqrt(9 + 25 + 64) = sqrt(98) = 7*sqrt(2)

**Alice [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Bob [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Alice [5]:** Wow, all three distances are equal! So PQ = QR = RP = 7*sqrt(2). That means these three points form an equilateral triangle. In a cube, three vertices that form an equilateral triangle must be such that each pair is connected by a face diagonal or a space diagonal? Wait, but in a cube, the face diagonal is s*sqrt(2) and space diagonal is s*sqrt(3), where s is the edge length.

**Alice [2]:** 1. Compute PQ: distance between P(7,12,10) and Q(8,8,1): sqrt((8-7)^2 + (8-12)^2 + (1-10)^2) = sqrt(1 + 16 + 81) = sqrt(98) = 7*sqrt(2)
**Bob [2]:** 1. Compute PQ: distance between P(7,12,10) and Q(8,8,1): sqrt((8-7)^2 + (8-12)^2 + (1-10)^2) = sqrt(1 + 16 + 81) = sqrt(98) = 7*sqrt(2)

**Alice [3]:** 2. Compute QR: distance between Q(8,8,1) and R(11,3,9): sqrt((11-8)^2 + (3-8)^2 + (9-1)^2) = sqrt(9 + 25 + 64) = sqrt(98) = 7*sqrt(2)
**Bob [3]:** 2. Compute QR: distance between Q(8,8,1) and R(11,3,9): sqrt((11-8)^2 + (3-8)^2 + (9-1)^2) = sqrt(9 + 25 + 64) = sqrt(98) = 7*sqrt(2)

**Alice [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Bob [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Alice [5]:** Wow, all three distances are equal! So PQ = QR = RP = 7*sqrt(2). That means these three points form an equilateral triangle. In a cube, three vertices that form an equilateral triangle must be such that each pair is connected by a face diagonal or a space diagonal? Wait, but in a cube, the face diagonal is s*sqrt(2) and space diagonal is s*sqrt(3), where s is the edge length.

**Alice [3]:** 2. Compute QR: distance between Q(8,8,1) and R(11,3,9): sqrt((11-8)^2 + (3-8)^2 + (9-1)^2) = sqrt(9 + 25 + 64) = sqrt(98) = 7*sqrt(2)

**Bob [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Alice [5]:** Wow, all three distances are equal! So PQ = QR = RP = 7*sqrt(2). That means these three points form an equilateral triangle. In a cube, three vertices that form an equilateral triangle must be such that each pair is connected by a face diagonal or a space diagonal? Wait, but in a cube, the face diagonal is s*sqrt(2) and space diagonal is s*sqrt(3), where s is the edge length.

**Bob [3]:** 2. Compute QR: distance between Q(8,8,1) and R(11,3,9): sqrt((11-8)^2 + (3-8)^2 + (9-1)^2) = sqrt(9 + 25 + 64) = sqrt(98) = 7*sqrt(2)

**Alice [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Bob [5]:** 4. Now, since all three distances are equal, they form an equilateral triangle. In a cube, the side length of the equilateral triangle formed by three vertices can be related to the edge length s of the cube. If the triangle is formed by three vertices where each pair is connected by a face diagonal, then the side length of the triangle would be s*sqrt(2). But if it's formed by a space diagonal, it would be s*sqrt(3). Wait, but here the side length is 7*sqrt(2), so if it's a face diagonal, then s*sqrt(2) = 7*sqrt(2) => s = 7. If it's a space diagonal, s*sqrt(3) = 7*sqrt(2) => s = 7*sqrt(2/3), which is not an integer, but the problem says it's a cube, so probably s is integer. So likely it's a face diagonal.



### Work in progress (others)

**Alice [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Bob [5]:** 4. Now, since all three distances are equal, they form an equilateral triangle. In a cube, the side length of the equilateral triangle formed by three vertices can be related to the edge length s of the cube. If<...>

**Bob [4]:** 3. Compute RP: distance between R(11,3,9) and P(7,12,10): sqrt((7-11)^2 + (12-3)^2 + (10-9)^2) = sqrt(16 + 81 + 1) = sqrt(98) = 7*sqrt(2)

**Alice [5]:** Wow, all three distances are equal! So PQ = QR =<...>





Wait, given the limited time, I have to give an answer right now. Considering all my previous attempts, I have to conclude that the final answer is \boxed{294}

Exact replay in free generation: True
All artifacts: /home/uskovev/async_llm/training_runs/parallel_thinking/20261101T115331740144Z
